<a href="https://colab.research.google.com/github/chaeee01/3DGS-Character-Generation-Pipeline/blob/feature%2Fsam2-preprocess/SAM2_try1_260705.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##구글 드라이브 연결 및 SAM2 패키지 설치

In [ ]:
# 구글 드라이브와 코랩 컴퓨터를 연결하는 코드
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .

Cloning into 'sam2'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 1107 (delta 10), reused 4 (delta 4), pack-reused 1093 (from 2)
Receiving objects: 100% (1107/1107), 134.85 MiB | 18.66 MiB/s, done.
Resolving deltas: 100% (385/385), done.
/content/sam2
Obtaining file:///content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 8.7 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
  Created wheel for SAM-2: filename=sam_2-1.0-0.editable-cp312-cp312-linux_x86_64.whl size=13850 sha256=d9558644f3c042c1ede790d69b08d62f28cc81e76

## 프레임 쪼개기

In [ ]:
import os
import cv2
import torch
from sam2.build_sam import build_sam2_video_predictor

# 1. ⚠️ 구글 드라이브에 올려둔 영상 경로 (방금 복사한 경로를 붙여넣으세요)
video_path = "/content/drive/MyDrive/zombie_project/zombie_sample.mp4"

# 2. 쪼개진 프레임들이 저장될 경로 (이것도 구글 드라이브로 지정하면 영구 저장됩니다!)
output_dir = "/content/drive/MyDrive/zombie_project/zombie_frames"
os.makedirs(output_dir, exist_ok=True)

# 영상 읽기 및 프레임 쪼개기 작업 시작
cam = cv2.VideoCapture(video_path)
frame_idx = 0
while True:
    ret, frame = cam.read()
    if not ret:
        break
    # 파일명을 00000.jpg 형태로 저장
    cv2.imwrite(os.path.join(output_dir, f"{frame_idx:05d}.jpg"), frame)
    frame_idx += 1
cam.release()
print(f"총 {frame_idx}개의 좀비 비디오 프레임 추출 및 구글 드라이브 저장 완료!")

# 3. SAM 2 가중치 파일 다운로드 및 세팅
sam2_checkpoint = "./checkpoints/sam2_hiera_tiny.pt"
model_cfg = "sam2_hiera_t.yaml"

!mkdir -p checkpoints
!wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt

predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint)

RuntimeError: You're likely running Python from the parent directory of the sam2 repository (i.e. the directory where https://github.com/facebookresearch/sam2 is cloned into). This is not supported since the `sam2` Python package could be shadowed by the repository name (the repository is also named `sam2` and contains the Python package in `sam2/sam2`). Please run Python from another directory (e.g. from the repo dir rather than its parent dir, or from your home directory) after installing SAM 2.

프레임 쪼개기까지 완료함. 아래부터 아직 실행 안 함

1단계: 동영상 상태 초기화하기

In [ ]:
# SAM 2 예측기 상태 초기화
inference_state = predictor.init_state(video_path=output_dir)

2단계: 첫 번째 프레임(0번)에서 좀비 위치 알려주기 (프롬프트 지정)

In [ ]:
import numpy as np

ann_frame_idx = 0  # 첫 번째 프레임
ann_obj_id = 1     # 우리가 추적할 좀비 오브젝트의 ID (1번)

# 좀비가 있을 법한 위치의 좌표 [X, Y] 지정 (픽셀 기준)
# 예: 가로 500픽셀, 세로 500픽셀 위치에 좀비가 있다고 가정
points = np.array([[500, 500]], dtype=np.float32)

# 1은 '이 위치에 객체가 있다(Positive)', 0은 '배경이다(Negative)'를 의미
labels = np.array([1], dtype=np.int32)

# 첫 프레임에 프롬프트(점) 추가
_, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)
print("0번 프레임에 좀비 타겟 좌표 지정 완료!")

3단계: 전체 1758개 프레임으로 마스크 전성(Propagate)하기

In [ ]:
# 전체 비디오에서 좀비 추적 및 마스크 생성 (가장 핵심적인 AI 추론 루프)
video_segments = {}
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }
print("전체 1758개 프레임에 대한 좀비 추적 및 마스크 생성 완료!")